# Large PDF Similarity (bill / policy PDFs)

Self-contained notebook for testing the **PDF → text → clean → chunk → embed → cosine** pipeline.

```
PDF bytes -> extract text -> clean -> chunk_text() -> model.encode() -> util.cos_sim()
```

| Library | Best for | Why |
|---|---|---|
| `pypdf` | Simple, well-formed digital PDFs | Pure Python, fast, no system deps |
| `pdfplumber` | Multi-column layouts, tables | Better column/word ordering |
| `pymupdf` (`fitz`) | Speed + tricky layouts | Fastest; great fallback |

**Gotchas with real bill PDFs:**
- **Scanned / image-only PDFs** extract *nothing* — they need OCR (`pytesseract` + `pdf2image`). Always check `len(text)` per page.
- Process **page-by-page** for big PDFs so memory stays flat, then chunk the cleaned full text.

In [ ]:
# === Imports + model (run once) ===
# pip install sentence-transformers pypdf pymupdf requests
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")  # small, fast, good baseline

In [ ]:
# === Chunk -> embed -> aggregate (handles long docs) ===
def chunk_text(text, words_per_chunk=150, overlap=30):
    """Split text into overlapping word-windows so we don't cut a sentence's meaning in half."""
    words = text.split()
    step = words_per_chunk - overlap
    chunks = [" ".join(words[i:i + words_per_chunk])
              for i in range(0, len(words), step)]
    return [c for c in chunks if c.strip()]

def doc_similarity(doc_a, doc_b, agg="max"):
    """Compare two long documents by chunking both and aggregating chunk-pair cosine scores.

    agg='max'  -> best-matching passage (good for 'is any section copied?')
    agg='mean' -> overall thematic closeness
    """
    chunks_a = chunk_text(doc_a)
    chunks_b = chunk_text(doc_b)
    emb_a = model.encode(chunks_a, convert_to_tensor=True)
    emb_b = model.encode(chunks_b, convert_to_tensor=True)
    sim_matrix = util.cos_sim(emb_a, emb_b)  # (len_a x len_b)
    if agg == "max":
        return float(sim_matrix.max()), len(chunks_a), len(chunks_b)
    return float(sim_matrix.mean()), len(chunks_a), len(chunks_b)

In [ ]:
# === PDF -> clean text helpers ===
import re
from pypdf import PdfReader

def pdf_to_text(source):
    """Extract text from a PDF given a file path or raw bytes, page by page.

    Returns (full_text, per_page_char_counts). A page with ~0 chars usually means
    it's a scanned image -> you'd need OCR for that page.
    """
    import io
    reader = PdfReader(source if isinstance(source, str) else io.BytesIO(source))
    pages = [(p.extract_text() or "") for p in reader.pages]
    return "\n".join(pages), [len(p) for p in pages]

def clean_bill_text(text):
    """Strip common Congressional-PDF noise before chunking."""
    # join hyphenated line breaks:  infra-\nstructure -> infrastructure
    text = re.sub(r"-\n", "", text)
    # drop standalone line-number / page markers like 'S1234' or bare line numbers
    text = re.sub(r"^\s*[SHE]?\d{1,5}\s*$", " ", text, flags=re.MULTILINE)
    # collapse whitespace/newlines into single spaces
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def pdf_doc_similarity(pdf_a, pdf_b, agg="max"):
    """Full pipeline: two PDFs (path or bytes) -> cleaned -> chunked -> aggregated similarity."""
    text_a = clean_bill_text(pdf_to_text(pdf_a)[0])
    text_b = clean_bill_text(pdf_to_text(pdf_b)[0])
    return doc_similarity(text_a, text_b, agg=agg)

## 1. Self-contained demo (build sample PDFs in memory)

No network needed — generates sample multi-page PDFs with `pymupdf`, extracts text, and compares.

In [ ]:
import fitz  # pymupdf, only used here to MAKE sample PDFs so the demo is runnable

def make_pdf_bytes(text, words_per_page=120):
    """Render text across multiple pages and return the PDF as bytes."""
    words = text.split()
    doc = fitz.open()
    for i in range(0, len(words), words_per_page):
        page = doc.new_page()
        page.insert_textbox(fitz.Rect(50, 50, 550, 750),
                            " ".join(words[i:i + words_per_page]), fontsize=11)
    data = doc.tobytes()
    doc.close()
    return data

# Two policy docs that share a funding clause but differ elsewhere.
bill_a = (
    "SEC. 2. The Secretary shall establish a grant program to fund solar and wind "
    "infrastructure in rural communities, prioritizing projects that create local jobs. "
) * 20 + "SEC. 3. Reporting requirements shall be submitted annually to Congress. " * 20

bill_b = (
    "Appropriations for highway maintenance and bridge repair are authorized through 2030. "
) * 20 + (
    "The Department shall fund solar and wind infrastructure in rural communities, "
    "prioritizing projects that create local jobs. "
) * 10

unrelated = "This Act concerns the regulation of commercial fishing quotas in coastal waters. " * 40

pdf_a = make_pdf_bytes(bill_a)
pdf_b = make_pdf_bytes(bill_b)
pdf_unrelated = make_pdf_bytes(unrelated)

# 1) Inspect extraction quality (catch scanned/empty pages).
text_a, page_chars = pdf_to_text(pdf_a)
print(f"Extracted {len(page_chars)} pages, chars/page = {page_chars}")
print(f"Total chars after extraction: {len(text_a)}")
print("Cleaned preview:", clean_bill_text(text_a)[:160], "...\n")

# 2) Run the full PDF -> similarity pipeline.
related, na, nb = pdf_doc_similarity(pdf_a, pdf_b, agg="max")
unrel, _, nu = pdf_doc_similarity(pdf_a, pdf_unrelated, agg="max")
print(f"[PDF_a vs PDF_b]   max chunk similarity = {related:0.3f}  (shared clause -> high)")
print(f"[PDF_a vs unrelated PDF] max chunk similarity = {unrel:0.3f}  (different topic -> low)")

## 2. Real bill PDFs from GovInfo

Downloads actual bill PDFs and runs the same pipeline. `version`: `ih` (introduced House), `rh`, `eh`, `es`, `enr`, ...

In [ ]:
# === Download real bill PDFs from GovInfo and compare ===
import requests

def govinfo_bill_pdf(congress, bill_type, number, version):
    """Fetch a bill PDF from GovInfo as bytes. version: ih, rh, eh, es, enr, ..."""
    pkg = f"BILLS-{congress}{bill_type}{number}{version}"
    url = f"https://www.govinfo.gov/content/pkg/{pkg}/pdf/{pkg}.pdf"
    r = requests.get(url, headers={"User-Agent": "influence-network-demo"}, timeout=60)
    r.raise_for_status()
    return r.content, url

# Example: compare two versions of the same bill (introduced vs engrossed/enrolled).
pdf_1, url_1 = govinfo_bill_pdf(117, "hr", 3684, "ih")   # Infrastructure Investment and Jobs Act, introduced
pdf_2, url_2 = govinfo_bill_pdf(117, "hr", 3684, "enr")  # ...as enrolled
print("Downloaded:")
print(" ", url_1, f"({len(pdf_1):,} bytes)")
print(" ", url_2, f"({len(pdf_2):,} bytes)")

txt_1, pages_1 = pdf_to_text(pdf_1)
txt_2, pages_2 = pdf_to_text(pdf_2)
print(f"\nPDF 1: {len(pages_1)} pages, {len(txt_1):,} chars")
print(f"PDF 2: {len(pages_2)} pages, {len(txt_2):,} chars")

score, c1, c2 = pdf_doc_similarity(pdf_1, pdf_2, agg="max")
print(f"\n[introduced vs enrolled] max chunk similarity = {score:0.3f}  ({c1} vs {c2} chunks)")